In [38]:
import os

In [39]:
def detect_source_type(path : str) -> str:
    _, ext = os.path.splitext(path.lower())
    return ext
  

In [40]:
ext = detect_source_type("data/text_samples/intro_rag.txt")
ext  # returns '.txt'

'.txt'

In [41]:
from os import path
 
from langchain_community.document_loaders import (
        PyPDFLoader,
        Docx2txtLoader,
        CSVLoader
    )
from langchain_community.document_loaders import JSONLoader
   
def load_file(path: str):
        ext = detect_source_type(path)
   
        if ext == ".pdf":
                return PyPDFLoader(path).load()
           
        elif ext == ".docx":
                    return Docx2txtLoader(path).load()
           
        elif ext == ".csv":
                    return CSVLoader(path).load()
           
        elif ext == ".json":
                return JSONLoader(path, jq_schema=".", text_content = False).load()
           
        elif ext == ".xlsx":
                    raise NotImplementedError("Le loader Excel sera géré dans la scène suivante.")
           
        else:
                raise ValueError(f"Format non supporté : {ext}")


In [42]:
docs = load_file("../data/word_files/Document_RAG_Word.docx")
 
print("nb_docs :", len(docs))
print("contenu du 1er docs :", docs[0].page_content[:200])
print("métadonnées du 1er docs :", docs[0].metadata)

nb_docs : 1
contenu du 1er docs : Procédure interne – Ingestion de documents Word pour un pipeline RAG

Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans
métadonnées du 1er docs : {'source': '../data/word_files/Document_RAG_Word.docx'}


In [43]:
docs = load_file("../data/csv_excel_files/tickets_support.csv")
 
print("nb_docs :", len(docs))
print("contenu du 1er docs :", docs[0].page_content[:200])
print("métadonnées du 1er docs :", docs[0].metadata)

nb_docs : 4
contenu du 1er docs : id_ticket: 101
client: ACME Corp
priorite: Haute
date_creation: 2024-02-10
description: Le client signale un problème d'accès à la plateforme après une mise à jour.
agent: Okan
métadonnées du 1er docs : {'source': '../data/csv_excel_files/tickets_support.csv', 'row': 0}


In [44]:
docs = load_file("../data/json_files/support_tickets.json")
 
print("nb_docs :", len(docs))
print("contenu du 1er docs :", docs[0].page_content[:200])
print("métadonnées du 1er docs :", docs[0].metadata)

nb_docs : 1
contenu du 1er docs : [{"ticket_id": 501, "client": "TechVision", "priority": "high", "category": "API", "agent": "Alice Dupont", "details": {"summary": "Probl\u00e8me de connexion \u00e0 l\u2019API", "steps": ["Test des i
métadonnées du 1er docs : {'source': '/home/lady/Documents/insta/ia-generative/data/json_files/support_tickets.json', 'seq_num': 1}


In [45]:
#verifier le cleaning sur un exemple pdf 
def clean_text_generic(text :str):
        import re
        text = re.sub(r"\s+", " ", text)
        return text.strip()


In [46]:
def clean_word_text(text: str):
    import re
    text = text.replace("•", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


In [47]:
def clean_pdf_text (text : str):
       import re
       text = re.sub(r"\s+", " ", text)
       text = re.sub(r"-\s+", "", text)
       text = re.sub(r"Page \d+ / \d+", "", text)
       return text.strip()

In [48]:
def clean_tabular_text(text :str):
        import re
        text = re.sub(r"\s+", " ", text)
        return text.strip()

In [49]:
def clean_json_text(text :str):
        import re
        text = re.sub(r"\s+", " ", text)
        return text.strip()

In [50]:
def clean_document(doc, ext):
    """Nettoie le contenu d'un document selon son extension."""
    if ext == ".pdf":
        doc.page_content = clean_pdf_text(doc.page_content)
    elif ext == ".docx":
        doc.page_content = clean_word_text(doc.page_content)
    elif ext in [".csv", ".xlsx"]:
        doc.page_content = clean_tabular_text(doc.page_content)
    elif ext == ".json":
        doc.page_content = clean_json_text(doc.page_content)
    else:
        doc.page_content = clean_text_generic(doc.page_content)
    return doc


## chunking automatique des documents

In [51]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_splitter(ext:str):
   
       # PDF -> textes longs, 500-1200 tokens
        if ext ==".pdf":
            return RecursiveCharacterTextSplitter (
                chunk_size = 1200,
                chunk_overlap = 150,
                separators= ["\n\n", "\n", ".", " ", ""]
            )
        # Word → texte modéré, 700-1000 tokens
        elif ext == ".docx":
            return RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=150,
                separators=["\n\n", "\n", ".", " ", ""]
            )
        # CSV / Excel / SQL → textes courts, peu de contexte\n",
        elif ext in [".csv", ".xlsx", ".sql"]:
            return RecursiveCharacterTextSplitter(
                chunk_size=600,
                chunk_overlap=80,
                separators=["\n", ".", " ", ""]
            )
         # JSON → textes hiérarchiques, souvent compactés\n",
        elif ext == ".json":
            return RecursiveCharacterTextSplitter(
                chunk_size=800,
               chunk_overlap=100,
                separators=["\n", ".", " ", ""]
            )
       # Fallback par défaut\n",
        else:
           return RecursiveCharacterTextSplitter(
              chunk_size=800,
               chunk_overlap=100
          )
 

In [52]:
def chunk_docs(docs, ext):
    splitter = get_splitter(ext)
    return splitter.split_documents(docs)


In [53]:
path = "../data/word_files/Document_RAG_Word.docx"
ext = detect_source_type(path)
docs = load_file(path)
cleaned = [clean_document(doc, ext) for doc in docs]
chunks = chunk_docs(cleaned, ext)

print("nb_chunks :", len(chunks))
print("contenu du 1er chunk :", chunks[0].page_content[:200])
print("métadonnées du 1er chunk :", chunks[0].metadata)


nb_chunks : 3
contenu du 1er chunk : Procédure interne – Ingestion de documents Word pour un pipeline RAG Ce document est conçu spécialement pour accompagner la narration de la section dédiée à l’ingestion des fichiers Word (.docx) dans 
métadonnées du 1er chunk : {'source': '../data/word_files/Document_RAG_Word.docx'}


In [57]:
def clean_docs(docs, path):
    ext = detect_source_type(path)
   
    # On parcourt chaque document de la liste retournée par load_file
    for doc in docs:
        if ext == ".pdf":
            doc.page_content = clean_pdf_text(doc.page_content)
        elif ext == ".docx":
            doc.page_content = clean_word_text(doc.page_content)
        elif ext in [".csv", ".xlsx"]:
            doc.page_content = clean_tabular_text(doc.page_content)
        elif ext == ".json":
            doc.page_content = clean_json_text(doc.page_content)
        else:
            doc.page_content = clean_text_generic(doc.page_content)
           
    return docs
 

In [58]:
class SmartUnifiedIngestionPipeline:
   
        def __init__(self):
            pass      
        def load(self, path: str):
            docs = load_file(path)
            return docs  
        def clean(self, docs, path: str):
            cleaned = clean_docs(docs, path)
            return cleaned  
        def chunk(self, docs, path: str):
           chunks = chunk_docs(docs, path)
           return chunks
        def process(self, path: str):
          docs = self.load(path)
          cleaned = self.clean(docs, path)
          chunks = self.chunk(cleaned, path)
          return chunks

In [61]:
# tester la classe SmartUnifiedIngestionPipeline sur un fichier word 
pipeline = SmartUnifiedIngestionPipeline()
chunks = pipeline.process("../data/word_files/Document_RAG_Word.docx")
print("nb_chunks :", len(chunks))

nb_chunks : 4


In [62]:
# tester la classe SmartUnifiedIngestionPipeline sur un fichier CSV
pipeline = SmartUnifiedIngestionPipeline()
chunks = pipeline.process("../data/csv_excel_files/tickets_support.csv")
print("nb_chunks :", len(chunks))

nb_chunks : 4


In [63]:
# tester la classe SmartUnifiedIngestionPipeline sur un fichier json
pipeline = SmartUnifiedIngestionPipeline()
chunks = pipeline.process("../data/json_files/support_tickets.json")
print("nb_chunks :", len(chunks))

nb_chunks : 3
